# Week 9 Phase 1.5 — forensic confirmation of the physics-informed h coordinate

This notebook is a readable result tour. It consumes the frozen Phase 1.5 artifacts and does not rerun expensive active learning.

## 1. Definitions and question

We test $h=P/\sqrt{VX\,LS^3}$. Repository provenance fixes **LS as Gaussian spot radius** and **ST as substrate temperature**. Bare h is dimensional; Gan's full Keyhole number adds absorptivity and material/thermal normalization.

In [1]:
from pathlib import Path
import json, pandas as pd
ROOT = Path.cwd().parents[1]
OUT = ROOT/'outputs'/'week9_phase1_5_h_physics_confirmation'
audit=json.loads((OUT/'data_and_protocol_audit.json').read_text())
{k:audit[k] for k in ['rows','keyhole','conduction','LS_definition','ST_definition','frozen_outer_runs']}

{'rows': 405,
 'keyhole': 73,
 'conduction': 332,
 'LS_definition': 'Gaussian laser spot radius r0; stored in metres',
 'ST_definition': 'substrate temperature',
 'frozen_outer_runs': 100}

## 2. Are the empirical exponents consistent with theory?

Coefficient ratios are fitted on unstandardized log features, then assessed by bootstrap and split stability. The careful wording is *consistent with*, not *discovered the law*.

In [2]:
pd.read_csv(OUT/'empirical_exponent_summary.csv')

,exponent,theory,estimate,ci_lower,ci_upper,theory_inside_ci
0,VX relative to P,-0.5,-0.517798,-0.659198,-0.381127,True
1,LS relative to P,-1.5,-1.444157,-1.994969,-1.074209,True


![Theory versus empirical exponents](../../outputs/week9_phase1_5_h_physics_confirmation/figures/01_theory_vs_empirical_exponents.png)

## 3. How much of the problem is one-dimensional?

The next table uses the exact 20×5 frozen grouped splits. Global ranking can be excellent while probability quality and boundary-specific recall still differ.

In [3]:
s=pd.read_csv(OUT/'static_model_summary.csv')
s[s.model.isin(['log_h_logistic','logistic_M1','logistic_M3','gpc_4d','gpc_5d_h_augmented'])][['model','subset','mean_roc_auc','mean_pr_auc','mean_balanced_accuracy','mean_keyhole_recall','mean_brier_score']]

,model,subset,mean_roc_auc,mean_pr_auc,mean_balanced_accuracy,mean_keyhole_recall,mean_brier_score
3,gpc_4d,B1_q20,0.919710,0.867253,0.815909,0.732327,0.112831
4,gpc_4d,B1_q30,0.952277,0.897455,0.848762,0.761333,0.082874
5,gpc_4d,full81,0.992587,0.968286,0.925087,0.866438,0.026593
6,gpc_5d_h_augmented,B1_q20,0.917175,0.859852,0.817016,0.740425,0.114433
7,gpc_5d_h_augmented,B1_q30,0.950920,0.891614,0.851691,0.771310,0.084090
8,gpc_5d_h_augmented,full81,0.992171,0.966363,0.927299,0.871918,0.027177
12,log_h_logistic,B1_q20,0.903899,0.804413,0.810431,0.709549,0.124431
13,log_h_logistic,B1_q30,0.937929,0.834347,0.842876,0.750197,0.092592
14,log_h_logistic,full81,0.989697,0.950055,0.922196,0.860959,0.029587
30,logistic_M1,B1_q20,0.897966,0.804805,0.823711,0.737980,0.124410


![Boundary difficulty](../../outputs/week9_phase1_5_h_physics_confirmation/figures/04_boundary_difficulty.png)

## 4. Does OOF screening survive?

The zones use predictions made while each simulation was held out. They describe retrospective simulator-domain screening, not a safety rule.

In [4]:
pd.read_csv(OUT/'screening_zone_summary.csv')

,zone,count,keyhole,conduction,false_negative_if_low_zone_screened_conduction,false_positive_if_high_zone_screened_keyhole,negative_predictive_value,positive_predictive_value
0,low_p_lt_0.05,295,1,294,1,0,0.99661,NaN
1,ambiguous_0.05_to_0.95,62,26,36,0,0,NaN,NaN
2,high_p_gt_0.95,48,46,2,0,2,NaN,0.958333


In [5]:
json.loads((OUT/'calibration_summary.json').read_text()) | {'reliability':'shown in figure'}

{'ECE': 0.018661745045046543,
 'ECE_definition': 'sum_b (n_b/N)*abs(observed_rate_b-mean_probability_b)',
 'MCE': 0.35895216537573194,
 'MCE_definition': 'maximum non-empty-bin absolute calibration gap',
 'OOF_Brier': 0.029425123358329402,
 'binning': '10 equal-width probability bins on [0,1]',
 'oof_unit': 'mean of 20 out-of-fold probabilities for each of 405 unique simulations',
 'reliability': 'shown in figure',
 'status': 'retrospective simulator-domain screening rule'}

![OOF screening and calibration](../../outputs/week9_phase1_5_h_physics_confirmation/figures/02_logh_distribution_and_oof_calibration.png)

## 5. Does h help active learning?

All acquisition models use only revealed labels. Fold-B1 and test labels are evaluation-only. The two evaluations separate *h-model quality* from *h-query policy quality for a fixed 4D GPC*. The h-only advantage is already visible at the shared budget-16 design, so it cannot be attributed purely to acquisition.

In [6]:
a=pd.read_csv(OUT/'active_AULC_contrasts.csv')
a[a.metric.eq('B1_q20_AULC_16_80')][['model','comparator','model_mean','comparator_mean','difference','ci_lower','ci_upper']]

,model,comparator,model_mean,comparator_mean,difference,ci_lower,ci_upper
0,gpc4_assisted_product,frozen_4d_margin,0.805473,0.81352,-0.008047,-0.015327,-0.000317
2,gpc4_assisted_product,matched_random_mean_of_30_paths,0.805473,0.77622,0.029253,0.021729,0.036693
4,gpc4_h_acquisition,frozen_4d_margin,0.800827,0.81352,-0.012693,-0.021756,-0.002955
6,gpc4_h_acquisition,matched_random_mean_of_30_paths,0.800827,0.77622,0.024607,0.016768,0.031991
8,gpc5_margin,frozen_4d_margin,0.815386,0.81352,0.001866,-0.003222,0.006595
10,gpc5_margin,matched_random_mean_of_30_paths,0.815386,0.77622,0.039166,0.030356,0.047963
12,h_model_h_acquisition,frozen_4d_margin,0.831737,0.81352,0.018217,0.011567,0.024619
14,h_model_h_acquisition,matched_random_mean_of_30_paths,0.831737,0.77622,0.055517,0.045305,0.065545


![Active-learning q20 curves](../../outputs/week9_phase1_5_h_physics_confirmation/figures/06_active_q20_learning_curves.png)

## 6. Final interpretation

- h is physically motivated and globally strong.
- It is not dimensionless or universally thresholded.
- Residual 4D structure matters near Fold-B1.
- The h-only model/policy composite is strong at low budget, but this does not isolate an acquisition benefit; h-selected labels make a 4D GPC worse than 4D Margin.
- The 5D embedding and uncertainty product do not beat canonical 4D Margin.
- A single additive physics-ridge + 4D residual prototype is the focused next method worth testing.